# BI Pipeline - SEC Filings ETL & ADF Automation
## Azure ML Notebook | MADSC301 Business Intelligence Assignment

In [ ]:
# Cell 1 - Install required packages
import subprocess
import sys

def install(package):
    subprocess.check_call([sys.executable, "-m", "pip", "install", package, "-q"])

install("kagglehub")
install("pandas")
install("numpy")
install("matplotlib")
install("seaborn")
install("openpyxl")

print("All packages installed successfully.")

In [ ]:
# Cell 2 - Data Ingestion: Download SEC Filings Dataset
import os
import kagglehub
from pathlib import Path

print("Starting data ingestion...")
path = kagglehub.dataset_download("finnhub/sec-filings")
print(f"Dataset downloaded to: {path}")

# List all files in the dataset
all_files = []
for root, _, files in os.walk(path):
    for file in files:
        all_files.append(os.path.join(root, file))

print(f"\nTotal files found: {len(all_files)}")
for f in all_files:
    print(f)

In [ ]:
# Cell 3 - Load and Clean Data with Pandas
import pandas as pd
import numpy as np

csv_files = [f for f in all_files if f.lower().endswith(".csv")]
print(f"CSV files found: {len(csv_files)}")

dataframes = []
for file in csv_files:
    try:
        df = pd.read_csv(file)
        df["source_file"] = os.path.basename(file)
        dataframes.append(df)
        print(f"Loaded: {os.path.basename(file)} | Shape: {df.shape}")
    except Exception as e:
        print(f"Error loading {file}: {e}")

if dataframes:
    raw_df = pd.concat(dataframes, ignore_index=True)
else:
    raise ValueError("No CSV files loaded from dataset.")

# Clean the data
cleaned_df = raw_df.copy()
cleaned_df = cleaned_df.drop_duplicates()
cleaned_df.columns = [col.strip().lower().replace(" ", "_") for col in cleaned_df.columns]

for col in cleaned_df.select_dtypes(include="object").columns:
    cleaned_df[col] = cleaned_df[col].astype(str).str.strip()

cleaned_df = cleaned_df.replace("nan", np.nan)
cleaned_df = cleaned_df.dropna(how="all")

print(f"\nRaw shape: {raw_df.shape}")
print(f"Cleaned shape: {cleaned_df.shape}")
print(f"\nColumns: {list(cleaned_df.columns)}")
print("\nPreview:")
cleaned_df.head()

In [ ]:
import ssl
import sys
import pymongo

print("Python:", sys.version)
print("OpenSSL:", ssl.OPENSSL_VERSION)
print("PyMongo:", pymongo.version)

import socket

try:
    print(socket.gethostbyname("cluster0.brcygqe.mongodb.net"))
    print("DNS OK")
except Exception as e:
    print(e)

    import socket

for host in [
    "ac-lvrdkol-shard-00-00.brcygqe.mongodb.net",
    "ac-lvrdkol-shard-00-01.brcygqe.mongodb.net",
    "ac-lvrdkol-shard-00-02.brcygqe.mongodb.net"
]:
    try:
        socket.create_connection((host,27017),timeout=10)
        print(host,"CONNECTED")
    except Exception as e:
        print(host,e)
    

In [ ]:
import matplotlib.pyplot as plt

print("=== BI KPI Summary ===")
print(f"Total records: {len(cleaned_df):,}")
print(f"Total columns: {len(cleaned_df.columns)}")
print(f"Null values per column:\n{cleaned_df.isnull().sum()[cleaned_df.isnull().sum() > 0]}")
print(f"\nData types:\n{cleaned_df.dtypes.value_counts()}")

# -------------------------------------------------------
# Filing Count by Source File
# -------------------------------------------------------

if "source_file" in cleaned_df.columns:

    filing_counts = cleaned_df["source_file"].value_counts()

    print(f"\nFiling counts by source:\n{filing_counts}")

    plt.figure(figsize=(12,5))

    filing_counts.plot(
        kind="bar",
        color="steelblue",
        edgecolor="black"
    )

    plt.title("SEC Filings Count by Source File")
    plt.xlabel("Source File")
    plt.ylabel("Record Count")

    plt.xticks(rotation=45, ha="right")

    plt.tight_layout()

    plt.savefig("output/filing_counts_by_source.png", dpi=150)

    plt.show()

# -------------------------------------------------------
# Missing Values Chart
# -------------------------------------------------------

missing_values = cleaned_df.isnull().sum()
missing_values = missing_values[missing_values > 0]

if len(missing_values) > 0:

    plt.figure(figsize=(12,6))

    missing_values.sort_values(ascending=False).plot(
        kind="bar",
        color="tomato",
        edgecolor="black"
    )

    plt.title("Missing Values by Column")
    plt.xlabel("Columns")
    plt.ylabel("Missing Values")

    plt.xticks(rotation=45, ha="right")

    plt.grid(axis="y", linestyle="--", alpha=0.4)

    plt.tight_layout()

    plt.savefig("output/missing_values_bar_chart.png", dpi=150)

    plt.show()

else:
    print("No missing values found.")

In [6]:
# Cell 6 - Save Outputs and ADF Pipeline Logging
import json
from datetime import datetime
from pathlib import Path

# Create output directory
Path("output").mkdir(exist_ok=True)

# Save cleaned data to CSV
cleaned_df.to_csv("output/processed_sec_filings.csv", index=False)
print("Saved: output/processed_sec_filings.csv")

# Save summary statistics
summary_stats = cleaned_df.describe(include="all").transpose()
summary_stats.to_csv("output/summary_statistics.csv")
print("Saved: output/summary_statistics.csv")

# Save to Excel with multiple sheets
with pd.ExcelWriter("output/sec_filings_report.xlsx", engine="openpyxl") as writer:
    cleaned_df.head(500).to_excel(writer, sheet_name="Cleaned Data", index=False)
    summary_stats.to_excel(writer, sheet_name="Summary Stats")
print("Saved: output/sec_filings_report.xlsx")

# ADF Pipeline run log
run_log = {
    "pipeline_name": "BI_SEC_Filings_Pipeline",
    "run_timestamp": datetime.utcnow().isoformat(),
    "status": "SUCCESS",
    "records_ingested": len(raw_df),
    "records_after_cleaning": len(cleaned_df),
    "duplicates_removed": len(raw_df) - len(cleaned_df),
    "output_files": [
        "output/processed_sec_filings.csv",
        "output/summary_statistics.csv",
        "output/sec_filings_report.xlsx",
        "output/filing_counts_by_source.png",
        "output/missing_values_heatmap.png"
    ],
    "orchestration": "Azure Data Factory - Schedule Trigger (Daily)",
    "compute": "Azure ML Compute Instance"
}

with open("output/adf_pipeline_run_log.json", "w") as f:
    json.dump(run_log, f, indent=4)

print("\n=== ADF Pipeline Run Log ===")
print(json.dumps(run_log, indent=4))
print("\nPipeline completed successfully.")